# 04 · Reevaluación del modelo de producción actual (sin reentrenar)
Modelo desplegado en la app (`modelo_S4_lstm.keras`: apilado 64/64, WFL α=1.5 β=100, LOOKBACK 360, HORIZON 20, solo S4 + cíclicas) sobre su **propio split** (2022–2026). Responde con evidencia a dos preguntas del especialista: **¿cuál es la precisión/tasa de falsas alarmas?** y **¿supera significativamente a la persistencia?** (bootstrap por días + Diebold–Mariano).

In [ ]:
import sys, warnings
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from s4.config import cargar_config, variante, ruta
import json, joblib, tensorflow as tf
from pathlib import Path
from s4.pipeline import preparar_datos, baselines, calibrar_umbral, evaluar_y_registrar
from s4.windows import preparar_particion
from s4 import evaluate as ev, stats
cfg = cargar_config('../config/s4_completo_2022_2026.yaml')
DEPLOY = Path(cfg["rutas"]["base"]) / "notebook/deploy"      # <-- carpeta de la app
conf = json.load(open(DEPLOY / "config.json"))
esc = joblib.load(DEPLOY / "SCALER_S4_app.pkl"); esc.columnas_ = conf["FEATURES_COLS"]
modelo = tf.keras.models.load_model(DEPLOY / "modelo_S4_lstm.keras", compile=False)
datos = preparar_datos(cfg, usar_precursores=False)

In [ ]:
L, H, F = conf["LOOKBACK"], conf["HORIZON"], conf["FEATURES_COLS"]
parts = {k: preparar_particion(k, datos[k], esc, F, L, H, hora_corte_ut=0) for k in ("val", "test")}
from s4.train import predecir
pred = {k: predecir(modelo, parts[k], esc) for k in parts}
base = {k: baselines(cfg, datos, {**parts, "train": None}, k) for k in parts}

In [ ]:
# 1) Umbral de alerta del modelo, calibrado en VALIDACIÓN (máximo HSS)
u_modelo, tabla_u = calibrar_umbral(cfg, parts["val"], pred["val"])
print("Umbral de alerta elegido en validación:", u_modelo); tabla_u

In [ ]:
# 2) Evaluación en TEST (una sola vez)
filas = [evaluar_y_registrar(cfg, parts["test"], base["test"]["Persistencia"], "Persistencia", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], base["test"]["Climatología diaria"], "Climatología", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], pred["test"], "LSTM apilado + WFL (u=0.6)", "reevaluacion"),
         evaluar_y_registrar(cfg, parts["test"], pred["test"], f"LSTM apilado + WFL (u={u_modelo})", "reevaluacion", umbral_alerta=u_modelo)]
cols = ["modelo","RMSE_global","RMSE_evento","RMSE_calma","sesgo_evento","POD","Precision","FAR","CSI","HSS","POD_inicio","antelacion_media_min"]
tabla = pd.DataFrame(filas)[cols]; tabla.to_csv(ruta(cfg,"reportes")/"tables/tabla_reevaluacion_produccion.csv", index=False); tabla.round(4)

In [ ]:
# 3) ¿En qué fase gana el modelo? (inicio vs sostenido)
pd.concat({"Persistencia": ev.metricas_por_fase(parts["test"], base["test"]["Persistencia"]),
           "LSTM": ev.metricas_por_fase(parts["test"], pred["test"], umbral_alerta=u_modelo)})

In [ ]:
# 4) Falsas alarmas: ¿son "casi eventos"? (pregunta del especialista)
ev.analisis_falsas_alarmas(parts["test"], pred["test"], umbral_alerta=u_modelo)

In [ ]:
# 5) Significancia frente a persistencia: bootstrap por DÍAS + Diebold–Mariano
p = parts["test"]; m = ~p.objetivo_reconstruido
args = (p.y_real, base["test"]["Persistencia"], pred["test"], p.dia)
print("RMSE evento :", stats.bootstrap_bloques(*args, stats.m_rmse_evento(), mascara=m))
print("RMSE global :", stats.bootstrap_bloques(*args, stats.m_rmse_global, mascara=m))
print("Diebold–Mariano (MSE):", stats.diebold_mariano(*args, mascara=m))

**Lectura:** diferencia > 0 con IC que no cruza 0 ⇒ la LSTM es significativamente mejor que la persistencia en esa métrica. Reportar ambas (evento y global): el trade-off es parte del resultado.